In [52]:
import pandas as pd

labels_path = "data/train_labels.csv"
sequences_path = "data/train_sequences.csv"

labels = pd.read_csv(labels_path).dropna()
sequences = pd.read_csv(sequences_path)
sequences.head()

,target_id,sequence,temporal_cutoff,description,all_sequences
0,1SCL_A,GGGUGCUCAGUACGAGAGGAACCGCACCC,1995-01-26,"THE SARCIN-RICIN LOOP, A MODULAR RNA",>1SCL_1|Chain A|RNA SARCIN-RICIN LOOP|Rattus n...
1,1RNK_A,GGCGCAGUGGGCUAGCGCCACUCAAAAGGCCCAU,1995-02-27,THE STRUCTURE OF AN RNA PSEUDOKNOT THAT CAUSES...,>1RNK_1|Chain A|RNA PSEUDOKNOT|null\nGGCGCAGUG...
2,1RHT_A,GGGACUGACGAUCACGCAGUCUAU,1995-06-03,24-MER RNA HAIRPIN COAT PROTEIN BINDING SITE F...,>1RHT_1|Chain A|RNA (5'-R(P*GP*GP*GP*AP*CP*UP*...
3,1HLX_A,GGGAUAACUUCGGUUGUCCC,1995-09-15,P1 HELIX NUCLEIC ACIDS (DNA/RNA) RIBONUCLEIC ACID,>1HLX_1|Chain A|RNA (5'-R(*GP*GP*GP*AP*UP*AP*A...
4,1HMH_E,GGCGACCCUGAUGAGGCCGAAAGGCCGAAACCGU,1995-12-07,THREE-DIMENSIONAL STRUCTURE OF A HAMMERHEAD RI...,">1HMH_1|Chains A, C, E|HAMMERHEAD RIBOZYME-RNA..."


In [53]:
labels.head()

,ID,resname,resid,x_1,y_1,z_1
0,1SCL_A_1,G,1,13.760,-25.974001,0.102
1,1SCL_A_2,G,2,9.310,-29.638000,2.669
2,1SCL_A_3,G,3,5.529,-27.813000,5.878
3,1SCL_A_4,U,4,2.678,-24.900999,9.793
4,1SCL_A_5,G,5,1.827,-20.136000,11.793


In [54]:
# ------- Examining the DataFrame Studying Pair Distances -------
import pandas as pd
import numpy as np

# Assuming your DataFrame 'labels' is already loaded
# Use only the first 20 residues for the example
coords = labels[['x_1', 'y_1', 'z_1']][100:110].values

# Calculate the pairwise Euclidean distance matrix.
diff = coords[:, None, :] - coords[None, :, :]
dist_matrix = np.sqrt(np.sum(diff**2, axis=-1))

# For each of the first 20 residues, find the closest residue (excluding self)
for i in range(len(coords)):
    distances = dist_matrix[i].copy()
    distances[i] = np.inf  # ignore self-distance
    closest_idx = np.argmin(distances)
    closest_distance = distances[closest_idx]
    
    # Get residue types from the DataFrame
    res_i_type = labels.iloc[i]['resname']
    res_j_type = labels.iloc[closest_idx]['resname']
    
    print(f"Residue {i} ({res_i_type}) is closest to residue {closest_idx} ({res_j_type}) with distance = {closest_distance:.3f} Å")


Residue 0 (G) is closest to residue 1 (G) with distance = 5.526 Å
Residue 1 (G) is closest to residue 0 (G) with distance = 5.526 Å
Residue 2 (G) is closest to residue 3 (U) with distance = 5.069 Å
Residue 3 (U) is closest to residue 2 (G) with distance = 5.069 Å
Residue 4 (G) is closest to residue 5 (C) with distance = 4.956 Å
Residue 5 (C) is closest to residue 4 (G) with distance = 4.956 Å
Residue 6 (U) is closest to residue 5 (C) with distance = 5.500 Å
Residue 7 (C) is closest to residue 8 (A) with distance = 5.594 Å
Residue 8 (A) is closest to residue 9 (G) with distance = 5.394 Å
Residue 9 (G) is closest to residue 8 (A) with distance = 5.394 Å


In [ ]:
import pandas as pd
import numpy as np

IGNORE_BASE_PAIRS = True

IDs = sequences['target_id'].values
pair_stats = {}

for ID in IDs:
    print(f"Processing ID: {ID}")
    # Filter by ID included in string in ID col and x_1 , y_1, z_1 from labels
    coords = labels[labels['ID'].str.contains(ID)]
    coords = coords[:50]

    # Create Distance Matrix Numpy Array
    coords = coords[['x_1', 'y_1', 'z_1']].values
    # Calculate the pairwise Euclidean distance matrix.
    diff = coords[:, None, :] - coords[None, :, :]
    dist_matrix = np.sqrt(np.sum(diff**2, axis=-1))
    # For each of the first 20 residues, find the closest residue (excluding self)


    for i in range(len(coords)):
        distances = dist_matrix[i].copy()
        distances[i] = np.inf  # ignore self-distance
        
        if IGNORE_BASE_PAIRS:
            if i > 2 and i < len(coords) - 3:
                distances[i-3] = np.inf  # ignore base pairs
                distances[i-2] = np.inf  
                distances[i-1] = np.inf
                distances[i+1] = np.inf
                distances[i+2] = np.inf
                distances[i+3] = np.inf
        
        closest_idx = np.argmin(distances)
        closest_distance = distances[closest_idx]
        
        # Get residue types from the DataFrame
        res_i_type = labels.iloc[i]['resname']
        res_j_type = labels.iloc[closest_idx]['resname']
        
        # type is not A C G or U replace with '-'
        if res_i_type not in ['A', 'C', 'G', 'U']:
            res_i_type = 'N'
        if res_j_type not in ['A', 'C', 'G', 'U']:
            res_j_type = 'N'
        
        pair_key = sorted([res_i_type, res_j_type])
        pair_key = tuple(pair_key)
        
        if (res_i_type, res_j_type)in pair_stats:
            if closest_distance > 9:
                # Ignore distances greater than 10
                continue
            
            pair_stats[pair_key]["count"] += 1
            pair_stats[pair_key]["total_distance"] += closest_distance
            pair_stats[pair_key]["min_distance"] = min(pair_stats[pair_key]["min_distance"], closest_distance)
            pair_stats[pair_key]["max_distance"] = max(pair_stats[pair_key]["max_distance"], closest_distance)
            pair_stats[pair_key]["avg_distance"] = pair_stats[pair_key]["total_distance"] / pair_stats[pair_key]["count"]
        else:
            pair_stats[pair_key] = {
                "count": 1,
                "total_distance": closest_distance,
                "min_distance": closest_distance,
                "max_distance": closest_distance,
                "avg_distance": closest_distance,
                "ignore_base_pairs": IGNORE_BASE_PAIRS
            }

# Convert to DataFrame
# turn keys into 1 string each
pair_stats = {f"{k[0]}-{k[1]}": v for k, v in pair_stats.items()}
pair_stats_df = pd.DataFrame.from_dict(pair_stats, orient='index')
# drop total_distance
pair_stats_df = pair_stats_df.drop(columns=["total_distance"])

pair_stats_df



Processing ID: 1SCL_A
Processing ID: 1RNK_A
Processing ID: 1RHT_A
Processing ID: 1HLX_A
Processing ID: 1HMH_E
Processing ID: 1RNG_A
Processing ID: 1MME_D
Processing ID: 1KAJ_A
Processing ID: 1SLO_A
Processing ID: 1BIV_A
Processing ID: 1ANR_A
Processing ID: 1ZIG_A
Processing ID: 1ZIH_A
Processing ID: 1ETF_A
Processing ID: 1ZIF_A
Processing ID: 1KPD_A
Processing ID: 1IKD_A
Processing ID: 1ZDI_S
Processing ID: 1AFX_A
Processing ID: 1EBQ_A
Processing ID: 1EBR_A
Processing ID: 1ULL_A
Processing ID: 1KIS_B
Processing ID: 1KIS_A
Processing ID: 1ATO_A
Processing ID: 1TLR_A
Processing ID: 1VOP_A
Processing ID: 1AQO_A
Processing ID: 1ATV_A
Processing ID: 1ATW_A
Processing ID: 1UUU_A
Processing ID: 1AUD_B
Processing ID: 2U2A_A
Processing ID: 1A4T_A
Processing ID: 1A60_A
Processing ID: 1A51_A
Processing ID: 2A9L_A
Processing ID: 1A1T_B
Processing ID: 1A9N_Q
Processing ID: 3PHP_A
Processing ID: 2TPK_A
Processing ID: 7MSF_S
Processing ID: 5MSF_S
Processing ID: 1LDZ_A
Processing ID: 1ZDK_S
Processing

,count,min_distance,max_distance,avg_distance
G-G,2171,3.171940,10.222204,7.634789
C-G,1,9.928677,9.928677,9.928677
C-U,1,6.809999,6.809999,6.809999
C-C,1577,3.685041,8.998447,7.307775
G-U,3,5.244127,8.603056,6.363770
A-U,2,8.795210,8.872931,8.834070
A-A,628,3.428841,8.995806,7.589609
A-C,1,9.739456,9.739456,9.739456
A-G,1,8.184795,8.184795,8.184795
U-U,64,4.861167,8.927059,7.549633
